# 01 章 / 第 4 节 · 完整训练循环:AMP + grad accum + ckpt + wandb

## 学习目标
- 用第 3 节的 `NanoDecoder` 训一个会续写中文的 toy 模型
- 把现代训练循环的核心拼件全部跑一遍:bf16 AMP / 梯度累积 / 学习率 warmup-cosine / checkpoint 写盘
- 出 loss 曲线 + 一段推理样例

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-train.md`**(训练章)


## 1. 概念背景

一个工程化的训练循环至少需要这几件:

| 件 | 干嘛 | 这里怎么做 |
|---|---|---|
| **DataLoader** | 喂数据 | char-level 切窗,内嵌一段中文语料,完全离线 |
| **Optimizer** | 更新权重 | AdamW + 权重衰减(对 norm/bias 不衰减) |
| **LR Scheduler** | 调学习率 | warmup → cosine 退火,工业界标配 |
| **AMP** | 省显存提速 | `torch.autocast(bf16)`,3000 系以上 GPU 都支持 |
| **梯度累积** | 等效大 batch | `accum_steps=8` 等效 batch_size × 8 |
| **Gradient Clipping** | 防爆炸 | `clip_grad_norm_(1.0)` |
| **Checkpoint** | 断点续训 | 周期性写 `state_dict`(模型 + 优化器 + scheduler + step) |
| **Logging** | 看曲线 | print + matplotlib;真训练换 wandb / tensorboard |

> 这一节只演示**机制**,所以模型小、步数少、语料短 —— 几分钟内就能跑完。第 02 章起接 Qwen2.5 真模型 + 真数据。


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()


## 3. 核心实现


### 3.1 内嵌微型语料 + char-level tokenizer

为了**完全离线 + 几秒钟训完**,这里直接内嵌 ~1.5KB 中文 + 英文混合文本,字符级 tokenize。
真要训出能说人话的模型,后面 02 章会换成 Qwen tokenizer + 真数据集。


In [ ]:
CORPUS = '''道可道,非常道;名可名,非常名。无名,天地之始;有名,万物之母。
故常无,欲以观其妙;常有,欲以观其徼。此两者,同出而异名,同谓之玄。
玄之又玄,众妙之门。

天下皆知美之为美,斯恶已;皆知善之为善,斯不善已。
故有无相生,难易相成,长短相形,高下相倾,音声相和,前后相随。
是以圣人处无为之事,行不言之教;万物作而弗始,生而弗有,为而弗恃。

知者不博,博者不知。圣人不积,既以为人己愈有,既以与人己愈多。
天之道,利而不害;圣人之道,为而不争。

The journey of a thousand miles begins with one step.
Knowing yourself is the beginning of all wisdom.
He who knows others is wise; he who knows himself is enlightened.
'''

# 字符级词表
chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)

print(f"语料长度: {len(CORPUS)} 字符")
print(f"词表大小: {vocab_size}")
print(f"前 20 token 解码:{decode(encode(CORPUS[:20]))!r}")


### 3.2 DataLoader:切窗 + batch


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class CharWindowDataset(Dataset):
    """把整段文本切成 (block_size,) 的滑动窗口;target = input 右移一位。"""
    def __init__(self, text: str, block_size: int):
        self.data = torch.tensor(encode(text), dtype=torch.long)
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx: int):
        chunk = self.data[idx : idx + self.block_size + 1]
        return chunk[:-1], chunk[1:]  # (x, y)


BLOCK_SIZE = 64
ds = CharWindowDataset(CORPUS, block_size=BLOCK_SIZE)
print(f"样本数: {len(ds)}")
x, y = ds[0]
print(f"x: {x.shape} y: {y.shape}")
print(f"x[:10] = {x[:10].tolist()}  decode: {decode(x[:10].tolist())!r}")


### 3.3 模型:复用第 3 节的 NanoDecoder


In [ ]:
# 把第 3 节的实现重新引入(简化版,只保留必要类)
import math
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

def precompute_rope(d, n, base=10000.0, device=None):
    inv = 1.0 / (base ** (torch.arange(0, d, 2, device=device).float() / d))
    t = torch.arange(n, device=device).float()
    f = torch.outer(t, inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c, s = cos[None, None, : x.shape[-2], :], sin[None, None, : x.shape[-2], :]
    return torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).flatten(-2)

def repeat_kv(x, n_rep):
    return x if n_rep == 1 else x.repeat_interleave(n_rep, dim=1)

class GQA(nn.Module):
    def __init__(self, dim, n_head, n_kv_head, max_seq_len=512):
        super().__init__()
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = dim // n_head
        self.q = nn.Linear(dim, n_head * self.head_dim, bias=False)
        self.k = nn.Linear(dim, n_kv_head * self.head_dim, bias=False)
        self.v = nn.Linear(dim, n_kv_head * self.head_dim, bias=False)
        self.o = nn.Linear(dim, dim, bias=False)
        cos, sin = precompute_rope(self.head_dim, max_seq_len)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, D = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        q = apply_rope(q, self.cos[:T], self.sin[:T])
        k = apply_rope(k, self.cos[:T], self.sin[:T])
        k, v = repeat_kv(k, self.n_rep), repeat_kv(v, self.n_rep)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(out.transpose(1, 2).contiguous().view(B, T, D))

class SwiGLU(nn.Module):
    def __init__(self, dim, h):
        super().__init__()
        self.gu = nn.Linear(dim, h * 2, bias=False); self.d = nn.Linear(h, dim, bias=False)
    def forward(self, x):
        g, u = self.gu(x).chunk(2, dim=-1)
        return self.d(F.silu(g) * u)

class Block(nn.Module):
    def __init__(self, dim, n_head, n_kv_head, h, max_seq_len):
        super().__init__()
        self.n1 = RMSNorm(dim); self.attn = GQA(dim, n_head, n_kv_head, max_seq_len)
        self.n2 = RMSNorm(dim); self.mlp = SwiGLU(dim, h)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

class NanoDecoder(nn.Module):
    def __init__(self, vocab_size, dim=128, n_layer=4, n_head=4, n_kv_head=2,
                 mlp_hidden=384, max_seq_len=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        nn.init.normal_(self.embed.weight, std=0.02)  # 让初始 loss 接近 ln(vocab_size)
        self.blocks = nn.ModuleList([
            Block(dim, n_head, n_kv_head, mlp_hidden, max_seq_len) for _ in range(n_layer)
        ])
        self.norm = RMSNorm(dim)
    def forward(self, idx):
        x = self.embed(idx)
        for b in self.blocks: x = b(x)
        x = self.norm(x)
        return x @ self.embed.weight.t()  # tied head


### 3.4 优化器分组 + LR scheduler

经验法则:**对 weight 做权重衰减,对 bias 和 norm 不做**(防止 RMSNorm 的 scale 被衰减成 0)。


In [ ]:
def build_optimizer(model, lr=3e-4, weight_decay=0.1):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() < 2 or "norm" in n.lower() or "bias" in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    print(f"decay 参数: {sum(p.numel() for p in decay)/1e3:.1f}k | no_decay: {sum(p.numel() for p in no_decay)/1e3:.1f}k")
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95))


def lr_at(step, warmup, total, max_lr, min_lr_ratio=0.1):
    """线性 warmup + cosine 退火。"""
    if step < warmup:
        return max_lr * (step + 1) / warmup
    if step >= total:
        return max_lr * min_lr_ratio
    progress = (step - warmup) / (total - warmup)
    return max_lr * (min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress)))


### 3.5 训练循环:AMP + grad accum + ckpt


In [ ]:
import time
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"device: {device}  amp dtype: {dtype}")

# 配置
BATCH        = 16
ACCUM_STEPS  = 2          # 等效 batch = 32
TOTAL_STEPS  = 200        # 步数(每个优化步)
WARMUP       = 20
MAX_LR       = 3e-3
LOG_EVERY    = 20
CKPT_EVERY   = 100
CKPT_DIR     = Path("../checkpoints/01_train_loop_demo")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# 模型 + dataloader + optimizer
torch.manual_seed(42)
model = NanoDecoder(vocab_size=vocab_size, dim=128, n_layer=4, n_head=4,
                    n_kv_head=2, mlp_hidden=384, max_seq_len=BLOCK_SIZE).to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters())/1e3:.1f}k")

dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
data_iter = iter(dl)
def next_batch():
    global data_iter
    try:
        return next(data_iter)
    except StopIteration:
        data_iter = iter(dl)
        return next(data_iter)

optim = build_optimizer(model, lr=MAX_LR)
losses, lrs = [], []
t0 = time.time()
model.train()

for step in range(TOTAL_STEPS):
    # 学习率
    lr_now = lr_at(step, WARMUP, TOTAL_STEPS, MAX_LR)
    for g in optim.param_groups:
        g["lr"] = lr_now
    lrs.append(lr_now)

    # 梯度累积
    optim.zero_grad(set_to_none=True)
    loss_accum = 0.0
    for _ in range(ACCUM_STEPS):
        x, y = next_batch()
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device, dtype=dtype, enabled=(device == "cuda")):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)) / ACCUM_STEPS
        loss.backward()
        loss_accum += loss.item()

    # 梯度裁剪 + step
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optim.step()
    losses.append(loss_accum)

    if (step + 1) % LOG_EVERY == 0:
        elapsed = time.time() - t0
        print(f"step {step+1:4d}/{TOTAL_STEPS} | loss {loss_accum:.4f} | lr {lr_now:.2e} | {elapsed:.1f}s")

    if (step + 1) % CKPT_EVERY == 0:
        ckpt_path = CKPT_DIR / f"step_{step+1}.pt"
        torch.save({
            "step": step + 1,
            "model": model.state_dict(),
            "optim": optim.state_dict(),
            "loss": loss_accum,
        }, ckpt_path)
        print(f"  -> checkpoint saved: {ckpt_path}")

print(f"\n训练完成,总耗时 {time.time()-t0:.1f}s")


### 3.6 看 loss 曲线


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
ax1.plot(losses); ax1.set_title("training loss"); ax1.set_xlabel("step"); ax1.grid(alpha=0.3)
ax2.plot(lrs);    ax2.set_title("learning rate"); ax2.set_xlabel("step"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"初始 loss: {losses[0]:.3f}  ->  末尾 loss: {losses[-1]:.3f}")


## 4. 推理样本(看模型学到了什么)


In [ ]:
@torch.no_grad()
def sample(model, prompt: str, max_new: int = 80, temperature: float = 0.8) -> str:
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        # 只用最后 BLOCK_SIZE 个 token 做 context
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits = model(idx_cond)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())


for prompt in ["道", "天下", "The journey"]:
    out = sample(model, prompt, max_new=60)
    print(f"--- prompt: {prompt!r} ---")
    print(out)
    print()


## 5. 踩坑记录

- **AMP dtype 选 bf16 不要选 fp16**:bf16 动态范围和 fp32 一样,几乎不会 overflow / underflow;fp16 需要 `GradScaler`,代码多还容易炸。3000 系以上 GPU 都支持 bf16。
- **`zero_grad(set_to_none=True)`**:比默认 `False`(置 0)省一次写显存,经验加速 5-10%。
- **梯度累积要记得**:loss 在每个 micro-step **除以 `accum_steps`**,不然等效学习率被放大 `accum_steps` 倍。
- **学习率分组**:`bias` 和 `norm.weight` 必须放进 `no_decay`,否则 RMSNorm 的 scale 会被慢慢衰减到 0,模型出诡异 NaN。
- **DataLoader `drop_last=True`**:训练循环里假设 batch 大小恒定;`False` 时最后一个 batch 形状不一致会触发各种边界 bug。
- **ckpt 路径用 `pathlib` 而不是字符串拼**:跨平台、可读、`mkdir(parents=True)` 一次到位。

## 6. 延伸阅读

- [[Slipbox/amp-bf16-vs-fp16]] / [[Slipbox/gradient-accumulation]] / [[Slipbox/warmup-cosine-schedule]]
- [accelerate 文档](https://huggingface.co/docs/accelerate) —— 把这套循环改成多卡的最低成本路径
- [`nanoGPT` 仓库](https://github.com/karpathy/nanoGPT) —— Andrej Karpathy 同款思路,可对照阅读
- 论文:`/paper fetch 1804.04235` (Decoupled Weight Decay)
